# 04 — Jaccard intra-partido

Umbral: **J ≥ 0.5 → alta convergencia temática**, **J < 0.5 → baja convergencia**.


In [ ]:
# Cargar datos: cambia URL o XLSX segun corresponda
from pathlib import Path
import pandas as pd, numpy as np
from agendapp.io import fetch_endpoint, load_xlsx_municipio
from agendapp.transform import matriz_concejal_tema, binarizar, perfil_partido, filtrar_min_instrumentos
from agendapp.indices import shannon_norm, cv_shannon, jaccard_pairwise_mean, party_correlation
from agendapp import viz

# Opcion A: endpoint Apps Script (descomentar y poner URL real)
# URL = "https://script.google.com/macros/s/AKfycb.../exec"
# data = fetch_endpoint(URL)
# df_inst = pd.DataFrame(data["instrumentos"])

# Opcion B: xlsx local (piloto Guarne)
XLSX = Path("../..") / "Guarne_DILIGENCIADO.xlsx"
d = load_xlsx_municipio(XLSX)
df_inst = d["instrumentos"]
df_inst["Partido / Movimiento"] = df_inst["Partido / Movimiento"].astype(str).str.strip().str.upper()
df_inst.shape


In [ ]:
M = matriz_concejal_tema(df_inst, col_tema='Tematica', roles=('Proponente', 'Ponente'))
asign = (df_inst[df_inst['Rol'].isin(['Proponente', 'Ponente'])]
    .groupby('ID_Concejal')['Partido / Movimiento']
    .agg(lambda s: s.value_counts().idxmax()))
filas = []
for partido, cids in asign.groupby(asign).groups.items():
    sub = M.loc[[c for c in cids if c in M.index]]
    if sub.shape[0] < 2: continue
    j = jaccard_pairwise_mean(binarizar(sub).values)
    filas.append({'partido': partido, 'n': sub.shape[0], 'jaccard': j})
j_df = pd.DataFrame(filas).set_index('partido')
j_df.sort_values('jaccard', ascending=False)


In [ ]:
viz.barras_jaccard_por_partido(j_df['jaccard'])
